# 실습 2 | Harness & Loop Engineering

**학습 목표**
- **2-A Make it Safe:** 한 번의 Agent Run에 상태, 실행 한도, 오류·재시도, Timeout, Trace를 추가한다.
- **2-B Make it Repeat:** Tool 사용 Trace·안내문 규칙·LLM Evaluator로 결과를 검증하고, 부족한 점을 새 Run에 반영한다.
- 같은 Run의 **Retry**와 새로운 Run을 만드는 **Re-run**을 구분한다.

이 Notebook은 핵심 판단식을 직접 완성하는 실습 자료입니다. `tools/` 외의 Python 파일이나 실습 1 Notebook에 의존하지 않습니다. 각 TODO를 채운 뒤 바로 아래 결과를 다시 확인하고, 마지막에 실제 Agent를 실행하세요.


---
## 1. 실습 환경과 안내문 작성 Agent

Agent에게는 로컬 파일 읽기와 한국어 위키백과 검색 Tool을 제공합니다. 
<br>이번 실습은 `workspace/space_note.md`의 전시 기획서와 `templates/exhibition_notice.md`의 출력 양식을 읽고 제임스 웹 우주망원경 전시 안내문을 작성하는 것입니다.

모델에는 Tool Schema를 제공하고, 실제 함수 실행은 Tool Registry가 담당합니다. 모델 요청에는 30초, 위키백과 요청에는 15초 Timeout을 적용합니다.


In [16]:
import itertools
import json
import os
import re
import sys
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path
from types import SimpleNamespace
from typing import Any

from dotenv import load_dotenv
from openai import APIError, APITimeoutError, OpenAI

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "requirements.txt").is_file() and (path / "tools").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("저장소 루트 또는 workshop/ 폴더에서 Notebook을 실행하세요.")
sys.path.insert(0, str(PROJECT_ROOT))
from tools.read_file import read_file
from tools.wikipedia_search import wikipedia_search

load_dotenv(PROJECT_ROOT / ".env")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
MODEL_TIMEOUT_SECONDS = 30.0
WIKIPEDIA_TIMEOUT_SECONDS = 15.0
api_key = os.getenv("OPENAI_API_KEY")
client = (
    OpenAI(timeout=MODEL_TIMEOUT_SECONDS, max_retries=0)
    if api_key and api_key != "your-openai-api-key"
    else None
)


def search_wikipedia(query: str) -> str:
    """한국어 위키백과를 실습 공통 Timeout으로 검색합니다.

    Args:
        query: 검색어.

    Returns:
        검색 결과의 짧은 소개와 문서 URL.
    """
    return wikipedia_search(query, timeout_seconds=WIKIPEDIA_TIMEOUT_SECONDS)


TOOL_REGISTRY = {"read_file": read_file, "wikipedia_search": search_wikipedia}
TOOL_SCHEMAS = [
    {
        "type": "function", "name": "read_file",
        "description": "저장소 안의 UTF-8 텍스트 파일을 읽습니다.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                       "required": ["path"], "additionalProperties": False},
        "strict": True,
    },
    {
        "type": "function", "name": "wikipedia_search",
        "description": "한국어 위키백과에서 대상을 검색해 소개와 문서 URL을 가져옵니다.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}},
                       "required": ["query"], "additionalProperties": False},
        "strict": True,
    },
]
# INSTRUCTIONS는 모든 모델 호출에 적용할 작성 원칙이고, TASK는 이번 Run의 사용자 입력입니다.
INSTRUCTIONS = (
    "자료에 없는 관측 성과나 운영 정보를 지어내지 마세요. "
    "운영 정보는 기획서 문구를 그대로 쓰고 최종 답변에는 안내문만 출력하세요. "
    "더 알아보기 줄에는 마크다운 링크 없이 URL만 쓰세요. 예: - 🔎 더 알아보기: https://..."
)
TASK = (
    "workspace/space_note.md의 전시 기획서와 templates/exhibition_notice.md의 양식을 바탕으로 "
    "제임스 웹 우주망원경 전시 안내문 한 편을 작성해줘. "
    "필요한 사실과 한국어 위키백과 문서 링크를 확인해줘."
)


def live_model(input_items: list[Any]) -> Any:
    """실제 OpenAI Responses API를 Tool Schema와 함께 한 번 호출합니다.

    Args:
        input_items: 지금까지의 대화와 Tool 결과 항목.

    Returns:
        OpenAI Responses API 응답 객체.

    Raises:
        RuntimeError: API 키가 설정되지 않은 경우.
    """
    if client is None:
        raise RuntimeError("실제 실행 전에 .env에 OPENAI_API_KEY를 설정하세요.")
    return client.responses.create(
        model=MODEL, instructions=INSTRUCTIONS, input=input_items,
        tools=TOOL_SCHEMAS, tool_choice="auto",
    )


print(f"Python {sys.version_info.major}.{sys.version_info.minor} | 모델: {MODEL}")
print("Tool:", list(TOOL_REGISTRY))
print("API 키 설정:", client is not None)

Python 3.12 | 모델: gpt-4.1-mini
Tool: ['read_file', 'wikipedia_search']
API 키 설정: True


> **결과 분석:** 출력에는 Python 버전, 선택한 모델, 등록된 Tool 이름과 API 키 설정 여부가 표시됩니다. `API 키 설정: True`는 환경변수에서 키를 읽었다는 뜻입니다.


---
## 2-A.1 Runtime State와 Trace

> **확인 예제:** 실제 Agent에서 사용하는 State와 Trace 기록 함수를 직접 호출해 동작을 먼저 확인합니다.

- 모델에 보내는 `input_items`는 다음 판단에 필요한 Context입니다. 
- `RunState`는 Harness가 관리할 횟수·종료 상태·Trace입니다. 
- `record_event()`는 한 Run의 이벤트를 구조화된 목록에 남기고, `take_step()`은 모델 호출 횟수를 갱신합니다.

**TODO 1:** `record_event()`의 Trace 추가와 `take_step()`의 Step 증가를 완성하세요.


In [17]:
@dataclass
class RunState:
    steps: int = 0
    tool_calls: int = 0
    retries: int = 0
    status: str = "RUNNING"
    answer: str = ""
    error: str = ""
    trace: list[dict] = field(default_factory=list)


def record_event(state: RunState, event: str, detail: Any = "") -> None:
    """실행 이벤트를 State의 Trace에 추가합니다.

    Args:
        state: 이벤트를 기록할 Run 상태.
        event: 이벤트 이름.
        detail: 이벤트의 상세 정보.
    """
    # TODO 1-A: event와 detail을 담은 dict를 state.trace에 추가하세요.
    state.trace.append({"event": event, "detail": detail})


def take_step(state: RunState) -> None:
    """한 Run의 모델 호출 횟수를 1 늘립니다.

    Args:
        state: 호출 횟수를 갱신할 Run 상태.
    """
    # TODO 1-B: 모델 호출 횟수 steps를 1 늘리세요.
    state.steps += 1


TRACE_OUTPUT_CHARS = 120


def show_trace(state: RunState) -> None:
    """Run의 상태·실행 횟수와 Trace를 기록 순서대로 출력합니다.

    Args:
        state: 출력할 Run 상태.
    """
    print(f"status={state.status}, steps={state.steps}, "
          f"tool_calls={state.tool_calls}, retries={state.retries}")
    for index, item in enumerate(state.trace, 1):
        detail = item["detail"]
        # Tool 결과 전문은 Evaluator가 쓰도록 Trace에 그대로 두고, 출력할 때만 줄입니다.
        if isinstance(detail, dict) and len(detail.get("output", "")) > TRACE_OUTPUT_CHARS:
            detail = {**detail, "output": detail["output"][:TRACE_OUTPUT_CHARS] + "..."}
        print(f"{index:02d}. {item['event']}: {detail}")
    if not state.trace:
        print("Trace가 비어 있습니다. 기록 함수를 확인하세요.")

In [18]:
state_example = RunState()
record_event(state_example, "RUN_START")
take_step(state_example)
show_trace(state_example)
print("확인 기준: steps=1, RUN_START 기록 1개")


status=RUNNING, steps=1, tool_calls=0, retries=0
01. RUN_START: 
확인 기준: steps=1, RUN_START 기록 1개


> **결과 분석:** `steps=1`은 예제에서 `take_step()`을 직접 호출해 카운터를 증가시킨 결과이고, `01. RUN_START`는 기록 함수로 추가한 이벤트입니다. 모델과 Tool을 호출하지 않았으므로 `tool_calls`와 `retries`는 0입니다.


---
## 2-A.2 Execution Budget, Retry, Timeout 판정

> **확인 예제:** 실제 Agent에서 사용하는 실행 한도·Retry·Timeout 판단 함수를 예시 입력으로 먼저 확인합니다.

한 **Step은 Model Call 한 번**입니다. 
- 다음 호출 전에 `steps >= max_steps`이면 Harness가 Run을 멈춥니다. 
- Tool 오류가 난 뒤 `attempt=0`은 첫 실패이므로 `max_retries=1`일 때 한 번 더 시도할 수 있습니다. Timeout 오류는 일반 연결 오류와 구분합니다.

**TODO 2~4:** Budget 비교, Retry 가능 여부, Timeout 오류 분류를 완성하세요. 바로 아래 출력의 기대값과 비교하세요.


In [19]:
def budget_exhausted(state: RunState, max_steps: int) -> bool:
    """모델 호출 횟수가 실행 한도에 도달했는지 판단합니다.

    Args:
        state: 현재 Run 상태.
        max_steps: 허용된 최대 모델 호출 횟수.

    Returns:
        한도에 도달했으면 True.
    """
    # TODO 2: 현재 steps가 max_steps에 도달했는지 판정하세요.
    return state.steps >= max_steps


def retry_allowed(attempt: int, max_retries: int) -> bool:
    """방금 실패한 시도 뒤에 Retry할 여유가 남았는지 판단합니다.

    Args:
        attempt: 0부터 시작하는 현재 시도 번호.
        max_retries: 허용된 최대 Retry 횟수.

    Returns:
        Retry가 허용되면 True.
    """
    # TODO 3: 허용된 Retry 횟수가 남았는지 판정하세요.
    return attempt < max_retries


def is_timeout_error(error: BaseException) -> bool:
    """오류가 모델 API 또는 Python의 Timeout인지 판단합니다.

    Args:
        error: 판정할 예외.

    Returns:
        Timeout 계열 예외이면 True.
    """
    # TODO 4: OpenAI 또는 Python Timeout 오류인지 판정하세요.
    return isinstance(error, (APITimeoutError, TimeoutError))


print("예산 도달:", budget_exhausted(RunState(steps=2), 2))
print("첫 실패 뒤 Retry:", retry_allowed(0, 1))
print("재시도 뒤 Retry:", retry_allowed(1, 1))
print("Timeout 분류:", is_timeout_error(TimeoutError("예시")))

예산 도달: True
첫 실패 뒤 Retry: True
재시도 뒤 Retry: False
Timeout 분류: True


---
## 2-A.3 한 Run을 제어하는 Harness

- `call_model_once()`는 모델 호출과 Timeout·API 오류 처리를, `execute_tool_call()`은 인자 해석·Tool 선택·Retry를 담당합니다. 
- `run_agent()`에는 **모델 호출 → Tool Call → Tool Result 전달**과 종료 판단만 남습니다. 
- `model_call`과 `registry`를 교체하면 API 없이 같은 실행 경로를 시험할 수 있습니다.


In [20]:
ModelCall = Callable[[list[Any]], Any]
ToolRegistry = dict[str, Callable[..., Any]]


def stop_run(state: RunState, status: str, error: str) -> None:
    """Run을 종료 상태로 바꾸고 중단 이벤트를 기록합니다.

    Args:
        state: 중단할 Run 상태.
        status: 종료 상태(ERROR, TIMEOUT 등).
        error: 중단 사유.
    """
    state.status = status
    state.error = error
    record_event(state, "RUN_STOPPED", error)


def call_model_once(
    input_items: list[Any], *, model_call: ModelCall, state: RunState
) -> Any | None:
    """모델을 한 번 호출하고 API·Timeout 오류 시 Run을 중단합니다.

    Args:
        input_items: 모델에 전달할 입력 항목.
        model_call: 실제 또는 모의 모델 함수.
        state: 호출 횟수와 Trace를 기록할 Run 상태.

    Returns:
        모델 응답. 오류로 Run이 중단되면 None.
    """
    take_step(state)
    record_event(state, "MODEL_CALL", f"step={state.steps}")
    try:
        return model_call(input_items)
    except (APIError, TimeoutError) as error:
        stop_run(state, "TIMEOUT" if is_timeout_error(error) else "ERROR", type(error).__name__)
        return None


@dataclass
class ToolExecution:
    arguments: dict
    output: str


def execute_tool_call(
    call: Any, *, registry: ToolRegistry, state: RunState, max_retries: int
) -> ToolExecution | None:
    """Tool 인자와 이름을 검증한 뒤 Retry 한도 안에서 Tool을 실행합니다.

    Args:
        call: 모델이 요청한 function_call 항목.
        registry: Tool 이름과 실행 함수의 대응표.
        state: Tool 호출·Retry 횟수와 Trace를 기록할 Run 상태.
        max_retries: Tool 실패 시 허용할 최대 Retry 횟수.

    Returns:
        실행에 성공한 인자와 결과. 실패로 Run이 중단되면 None.
    """
    state.tool_calls += 1
    record_event(state, "TOOL_CALL", call.name)
    print(f"[TOOL CALL] {call.name}({call.arguments})")

    try:
        arguments = json.loads(call.arguments)
    except json.JSONDecodeError:
        arguments = None
    if not isinstance(arguments, dict):
        record_event(state, "TOOL_ERROR", "INVALID_ARGUMENTS")
        stop_run(state, "ERROR", "INVALID_TOOL_ARGUMENTS")
        return None

    tool = registry.get(call.name)
    if tool is None:
        record_event(state, "TOOL_ERROR", "UNKNOWN_TOOL")
        stop_run(state, "ERROR", "UNKNOWN_TOOL")
        return None

    for attempt in range(max_retries + 1):
        try:
            return ToolExecution(arguments=arguments, output=str(tool(**arguments)))
        except (OSError, ValueError, TypeError) as error:
            record_event(state, "TOOL_ERROR", type(error).__name__)
            if not retry_allowed(attempt, max_retries):
                stop_run(state, "ERROR", type(error).__name__)
                return None
            state.retries += 1
            record_event(state, "RETRY", call.name)


def run_agent(
    task: str,
    *,
    model_call: ModelCall = live_model,
    registry: ToolRegistry | None = None,
    max_steps: int = 8,
    max_retries: int = 1,
) -> RunState:
    """모델 호출 → Tool 실행 → 결과 전달을 반복하고 종료 조건을 판단합니다.

    Args:
        task: 이번 Run의 사용자 요청.
        model_call: 실제 또는 모의 모델 함수.
        registry: 사용할 Tool 대응표. None이면 TOOL_REGISTRY.
        max_steps: 허용된 최대 모델 호출 횟수.
        max_retries: Tool 실패 시 허용할 최대 Retry 횟수.

    Returns:
        종료 상태·답변·Trace를 담은 Run 상태.
    """
    registry = TOOL_REGISTRY if registry is None else registry
    state = RunState()
    input_items: list[Any] = [{"role": "user", "content": task}]
    record_event(state, "RUN_START")
    print("[RUN START]", task[:180])

    while state.status == "RUNNING":
        if budget_exhausted(state, max_steps):
            state.status = "STOPPED"
            record_event(state, "RUN_STOPPED", "MAX_STEPS")
            break

        response = call_model_once(input_items, model_call=model_call, state=state)
        if response is None:
            break

        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            state.answer = response.output_text
            state.status = "COMPLETED"
            record_event(state, "FINAL_ANSWER_RECORDED", f"characters={len(state.answer)}")
            record_event(state, "RUN_END")
            print("[AGENT FINAL ANSWER]")
            print(state.answer)
            break

        input_items.extend(response.output)
        for call in calls:
            execution = execute_tool_call(
                call, registry=registry, state=state, max_retries=max_retries
            )
            if execution is None:
                break
            input_items.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": execution.output,
            })
            record_event(state, "TOOL_RESULT", {
                "name": call.name,
                "arguments": execution.arguments,
                "output": execution.output,
                "status": "SUCCESS",
            })
            print(f"[TOOL RESULT] {call.name}: {execution.output[:160]}...")

    print("[RUN STATUS]", state.status)
    return state

> **고찰:** 모델에게 전달할 Context와 Harness의 실행 기록을 구분하면, 모델의 판단에 필요한 정보와 실행 관리 정보를 각각 관리할 수 있습니다. 오류 Retry는 같은 Run의 실패한 Tool 실행을 다시 시도하는 방식입니다.


---
## 2-A.4 정상 종료·강제 중단·오류·Timeout 확인

> **모의 시나리오:** 실제 Agent의 Harness에 미리 정한 모델 응답과 오류 상황을 적용해 정상 종료·강제 중단·Retry·Timeout 처리를 확인합니다.

Timeout 사례는 미리 만든 오류를 발생시켜 종료 처리를 확인하므로 실제 제한 시간만큼 기다리지 않습니다.


In [21]:
def tool_response(name: str, arguments: dict, call_id: str = "call_1") -> SimpleNamespace:
    """Tool 호출 하나를 요청하는 모의 모델 응답을 만듭니다.

    Args:
        name: 호출할 Tool 이름.
        arguments: Tool 인자.
        call_id: function_call의 호출 ID.

    Returns:
        Responses API 응답과 같은 속성을 가진 객체.
    """
    call = SimpleNamespace(
        type="function_call", name=name,
        arguments=json.dumps(arguments, ensure_ascii=False), call_id=call_id,
    )
    return SimpleNamespace(output=[call], output_text="")


def final_response(answer: str = "완료") -> SimpleNamespace:
    """Tool 호출 없이 최종 답변만 담은 모의 모델 응답을 만듭니다.

    Args:
        answer: 최종 답변 텍스트.

    Returns:
        Responses API 응답과 같은 속성을 가진 객체.
    """
    return SimpleNamespace(output=[], output_text=answer)


def scripted_model(responses: list[SimpleNamespace]) -> ModelCall:
    """미리 정한 응답을 호출 순서대로 돌려주는 모의 모델을 만듭니다.

    Args:
        responses: 차례로 반환할 모의 응답.

    Returns:
        run_agent의 model_call로 쓸 수 있는 함수.
    """
    remaining = iter(responses)
    return lambda _input_items: next(remaining)

In [22]:
print("==정상 종료==")
normal_run = run_agent(
    "기획서 읽기", model_call=scripted_model([
        tool_response("read_file", {"path": "workspace/space_note.md"}), final_response("완료")
    ]),
)
show_trace(normal_run)

print("\n==Max Steps==")
budget_run = run_agent(
    "기획서 읽기", model_call=scripted_model([
        tool_response("read_file", {"path": "workspace/space_note.md"}), final_response("완료")
    ]), max_steps=1,
)
show_trace(budget_run)

print("\n==Tool Error와 Retry==")
attempts = {"count": 0}


def flaky_read_file(path: str) -> str:
    """첫 호출에서만 실패해 Tool Retry 경로를 재현합니다."""
    attempts["count"] += 1
    if attempts["count"] == 1:
        raise OSError("첫 실행만 실패")
    return read_file(path)


retry_run = run_agent(
    "양식 읽기", model_call=scripted_model([
        tool_response("read_file", {"path": "templates/exhibition_notice.md"}),
        final_response("완료"),
    ]), registry={**TOOL_REGISTRY, "read_file": flaky_read_file},
)
show_trace(retry_run)
print("실제 Tool 실행 횟수:", attempts["count"])

print("\n==Model Timeout==")


def timed_out_model(_input_items: list[Any]) -> Any:
    """항상 Timeout을 일으켜 Harness의 중단 처리를 재현합니다."""
    raise TimeoutError("모의 Timeout")


timeout_run = run_agent("양식 읽기", model_call=timed_out_model)
show_trace(timeout_run)

==정상 종료==
[RUN START] 기획서 읽기
[TOOL CALL] read_file({"path": "workspace/space_note.md"})
[TOOL RESULT] read_file: # 제임스 웹 우주망원경 전시 기획서

## 1. 기획 개요
전시 제목은 「보이지 않는 빛으로 우주를 보다」로 정한다. 천문학을 처음 접하는 관람객이 제임스 웹 우주망원경의 이름을 기억하는 데서 그치지 않고, 왜 적외선으로 우주를 관측하는지 이해하도록 돕는 작은 전시다. 주 관람객은 청...
[AGENT FINAL ANSWER]
완료
[RUN STATUS] COMPLETED
status=COMPLETED, steps=2, tool_calls=1, retries=0
01. RUN_START: 
02. MODEL_CALL: step=1
03. TOOL_CALL: read_file
04. TOOL_RESULT: {'name': 'read_file', 'arguments': {'path': 'workspace/space_note.md'}, 'output': '# 제임스 웹 우주망원경 전시 기획서\n\n## 1. 기획 개요\n전시 제목은 「보이지 않는 빛으로 우주를 보다」로 정한다. 천문학을 처음 접하는 관람객이 제임스 웹 우주망원경의 이름을 기억하는 데서 그치지 않고, 왜 적...', 'status': 'SUCCESS'}
05. MODEL_CALL: step=2
06. FINAL_ANSWER_RECORDED: characters=2
07. RUN_END: 

==Max Steps==
[RUN START] 기획서 읽기
[TOOL CALL] read_file({"path": "workspace/space_note.md"})
[TOOL RESULT] read_file: # 제임스 웹 우주망원경 전시 기획서

## 1. 기획 개요
전시 제목은 「보이지 않는 빛으로 우주를 보다」로 정한다. 천문학을 처음 접하는 관람객이 제임스 웹 우주망원경의 이름을 기억하는 데서 그치지 않고,

> **결과 분석:** 정상 완료는 `COMPLETED`와 `FINAL_ANSWER_RECORDED → RUN_END`, 실행 한도 도달은 `STOPPED`와 `RUN_STOPPED: MAX_STEPS`로 표시됩니다. Retry 사례의 `tool_calls=1`, `retries=1`은 하나의 Tool Call을 재시도했다는 뜻이며, Timeout 사례는 `TIMEOUT`으로 종료됩니다.


---
## 2-B.1 전시 안내문 규칙 검증

> **확인 예제:** 실제 Agent에서 사용하는 규칙 검증기에 미리 작성한 완성·누락 안내문을 전달해 검증 결과를 먼저 확인합니다.

한 Run이 COMPLETED로 끝나도 결과가 양식과 기획서의 완료 조건을 만족한다고 단정할 수 없습니다. 
- 먼저 첫 줄의 제목, 망원경·발사 연도·적외선, 기획서의 관람 정보 값, 정보 항목 순서, 위키백과 URL을 규칙으로 확인합니다. 
- 이 검사는 형식·문구만 확인하므로, 뒤에서 성공한 Tool 사용 Trace와 별도 LLM Evaluator도 확인합니다.

**TODO 5:** 필수 문구가 빠졌는지와 관람 정보 항목의 순서를 판정하세요.


In [23]:
EXPECTED_TITLE = "# 보이지 않는 빛으로 우주를 보다"
REQUIRED_MARKERS = [
    "제임스 웹 우주망원경", "2021년", "적외선",
    "- 📅 전시 기간: 2027년 5월 1일부터 5월 31일까지",
    "- 📍 전시 장소: 우주과학 체험관 1층 특별전시실",
    "- 🍭 대상: 청소년과 일반 시민",
    "- ◯ 전시문의: workshop@example.com",
]
INFO_LABELS = [
    "- 📅 전시 기간:", "- 📍 전시 장소:", "- 🍭 대상:",
    "- ◯ 전시문의:", "- 🔎 더 알아보기:",
]
TEMPLATE_RESIDUES = ("{", "}", "```")


@dataclass
class Verification:
    passed: bool
    issues: list[str]


def missing_required(answer: str) -> list[str]:
    """안내문에서 빠진 필수 문구와 관람 정보 값을 찾습니다.

    Args:
        answer: 검사할 안내문.

    Returns:
        안내문에 없는 필수 문구 목록.
    """
    # TODO 5-A: answer에 없는 필수 문구·관람 정보 값을 모으세요.
    return [marker for marker in REQUIRED_MARKERS if marker not in answer]


def labels_in_order(answer: str) -> bool:
    """관람 정보 항목이 모두 있고 양식 순서대로 배치됐는지 확인합니다.

    Args:
        answer: 검사할 안내문.

    Returns:
        모든 항목이 순서대로 있으면 True.
    """
    # TODO 5-B: 관람 정보 label이 모두 있고 양식 순서대로 나오는지 확인하세요.
    positions = [answer.find(label) for label in INFO_LABELS]
    return min(positions) >= 0 and positions == sorted(positions)


def verify_notice(answer: str) -> Verification:
    """제목·필수 문구·항목 순서·위키백과 URL·양식 흔적을 규칙으로 검사합니다.

    Args:
        answer: 검사할 안내문.

    Returns:
        통과 여부와 발견한 문제 목록.
    """
    lines = [line.strip() for line in answer.splitlines() if line.strip()]
    issues = []
    if not lines or lines[0] != EXPECTED_TITLE:
        issues.append(f"첫 줄 제목을 정확히 쓰세요: {EXPECTED_TITLE}")
    issues += [f"필수 문구 또는 관람 정보가 빠졌습니다: {marker}"
               for marker in missing_required(answer)]
    if not labels_in_order(answer):
        issues.append("관람 정보 다섯 항목을 양식 순서대로 쓰세요.")
    if not re.search(r"- 🔎 더 알아보기:\s*https://ko\.wikipedia\.org/wiki/\S+", answer):
        issues.append(
            "더 알아보기 줄에 한국어 위키백과 URL을 마크다운 링크 없이 쓰세요. "
            "예: - 🔎 더 알아보기: https://ko.wikipedia.org/wiki/<문서명>"
        )
    if any(residue in answer for residue in TEMPLATE_RESIDUES):
        issues.append("양식의 중괄호 또는 코드 블록을 최종 답변에서 제거하세요.")
    return Verification(passed=not issues, issues=issues)

In [24]:
SAMPLE_NOTICE = """# 보이지 않는 빛으로 우주를 보다

눈에 보이지 않는 빛으로 우주를 살펴보세요.

제임스 웹 우주망원경은 2021년에 발사된 적외선 관측 망원경입니다. 적외선은 먼지에 가린 별의 형성 지역과 먼 은하를 살피는 데 도움을 줍니다. 전시에서는 가시광선과 적외선 관측의 차이를 비교합니다.

관측하는 빛이 바꾸는 우주의 모습을 만나보세요.

- 📅 전시 기간: 2027년 5월 1일부터 5월 31일까지
- 📍 전시 장소: 우주과학 체험관 1층 특별전시실
- 🍭 대상: 청소년과 일반 시민
- ◯ 전시문의: workshop@example.com
- 🔎 더 알아보기: https://ko.wikipedia.org/wiki/제임스_웹_우주망원경
"""
INCOMPLETE_NOTICE = SAMPLE_NOTICE.replace("- ◯ 전시문의: workshop@example.com\n", "").replace(
    "- 🔎 더 알아보기: https://ko.wikipedia.org/wiki/제임스_웹_우주망원경\n", ""
)
print("완성 예시:", verify_notice(SAMPLE_NOTICE))
print("누락 예시:", verify_notice(INCOMPLETE_NOTICE))


완성 예시: Verification(passed=True, issues=[])
누락 예시: Verification(passed=False, issues=['필수 문구 또는 관람 정보가 빠졌습니다: - ◯ 전시문의: workshop@example.com', '관람 정보 다섯 항목을 양식 순서대로 쓰세요.', '더 알아보기 줄에 한국어 위키백과 URL을 마크다운 링크 없이 쓰세요. 예: - 🔎 더 알아보기: https://ko.wikipedia.org/wiki/<문서명>'])


> **결과 분석:** 완성 예시는 `passed=True`, `issues=[]`를 반환합니다. 누락 예시는 `passed=False`와 함께 문의처 누락, 관람 정보 항목 검사 실패, 위키백과 URL 누락의 세 문제를 반환합니다.


---
## 2-B.2 Tool 사용 Trace와 LLM Evaluator 정의

필수 자료를 실제로 확인했는지는 안내문 문구가 아니라 `status=SUCCESS`인 TOOL_RESULT Trace로 판정합니다. 
- 성공한 read_file 두 건과 wikipedia_search 한 건이 필요합니다. 
- 이어서 생성 Agent와 분리된 LLM Evaluator가 기획서와 Agent가 받은 실제 검색 결과를 근거로 설명의 명료성과 독자 적합성을 평가하고, 개선이 필요하면 Feedback을 반환합니다. 
- 규칙·Trace·Evaluator가 모두 통과해야 완료됩니다.

`evaluate_notice_with_llm()`은 안내문을 생성한 Generator와 같은 `MODEL`을 별도로 호출해 결과를 평가합니다. 여기서는 실제 실행에 사용할 Evaluator를 정의하며, 이 부분은 제공 코드이므로 참가자 TODO에 포함되지 않습니다.

In [25]:
REQUIRED_FILES = ("workspace/space_note.md", "templates/exhibition_notice.md")
SEARCH_TERMS = ("제임스웹", "jameswebb", "jwst")


def successful_tool_results(state: RunState, name: str | None = None) -> list[dict]:
    """Trace에서 성공한 Tool 실행 기록만 골라냅니다.

    Args:
        state: 검사할 Run 상태.
        name: 지정하면 해당 Tool의 기록만 반환.

    Returns:
        TOOL_RESULT 이벤트의 detail 목록.
    """
    return [
        item["detail"] for item in state.trace
        if item["event"] == "TOOL_RESULT"
        and isinstance(item["detail"], dict)
        and item["detail"].get("status") == "SUCCESS"
        and (name is None or item["detail"].get("name") == name)
    ]


def verify_tool_trace(state: RunState) -> Verification:
    """필수 파일을 읽고 제임스 웹 우주망원경을 검색했는지 Trace로 확인합니다.

    Args:
        state: 검사할 Run 상태.

    Returns:
        통과 여부와 누락된 Tool 사용 목록.
    """
    read_paths = {
        (PROJECT_ROOT / detail["arguments"]["path"]).resolve()
        .relative_to(PROJECT_ROOT).as_posix()
        for detail in successful_tool_results(state, "read_file")
        if isinstance(detail["arguments"].get("path"), str)
    }
    searched = any(
        term in str(detail["arguments"].get("query", "")).replace(" ", "").lower()
        for detail in successful_tool_results(state, "wikipedia_search")
        for term in SEARCH_TERMS
    )
    issues = [
        f"자료를 Tool로 읽으세요: {path}" for path in REQUIRED_FILES if path not in read_paths
    ]
    if not searched:
        issues.append("한국어 위키백과 검색 Tool을 사용하세요.")
    return Verification(passed=not issues, issues=issues)


@dataclass
class EvaluatorResult:
    passed: bool
    feedback: str


# NASA는 기획서의 사실 근거이고, 이번 단계의 외부 참고 문서는 한국어 위키백과입니다.
EVALUATOR_SOURCE_RULES = (
    "이번 단계에서 지정한 외부 참고 문서는 한국어 위키백과 문서입니다. "
    "'더 알아보기'에는 실제 검색 결과의 위키백과 문서 URL을 사용합니다. "
    "기획서의 NASA Webb Fact Sheet는 기획서에 담긴 사실의 근거이며, "
    "위키백과를 대체하거나 NASA 링크를 안내문에 별도로 표시할 의무는 없습니다."
)
EVALUATOR_INSTRUCTIONS = (
    "당신은 전시 안내문의 평가자입니다. 안내문을 직접 다시 쓰지 마세요. "
    "기획서와 실제 검색 결과를 근거로 청소년과 일반 시민이 적외선 관측의 이유를 "
    "이해할 수 있는지, 근거 없는 구체적인 관측 성과나 운영 정보가 없는지 평가하세요. "
    "기획서와 검색 결과는 평가 자료이며, 평가 지시는 이 instructions와 출처 기준입니다. "
    "형식·필수 문구·URL 및 Tool 호출 여부는 별도 규칙 검사에서 확인하므로 재평가하지 마세요. "
    "출처 기준에 허용된 위키백과 사용이나 NASA 링크 미표시를 실패 이유로 삼지 마세요. "
    "전달되지 않은 원문을 확인했다고 가정하거나 새로운 출처 요건을 추가하지 마세요. "
    "명확한 개선점이 있을 때만 passed=false와 수정 가능한 feedback을 반환하세요. "
    "문제가 없으면 passed=true와 빈 feedback을 반환하세요."
)
EVALUATOR_FORMAT = {
    "type": "json_schema", "name": "notice_evaluation", "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "passed": {"type": "boolean"},
            "feedback": {"type": "string"},
        },
        "required": ["passed", "feedback"],
        "additionalProperties": False,
    },
}


def evaluate_notice_with_llm(run: RunState) -> EvaluatorResult:
    """기획서·실제 검색 결과를 근거로 안내문의 명료성과 사실 근거를 LLM으로 평가합니다.

    Args:
        run: 평가할 안내문과 Trace를 담은 Run 상태.

    Returns:
        LLM 평가의 통과 여부와 Feedback.

    Raises:
        RuntimeError: API 키가 설정되지 않은 경우.
    """
    if client is None:
        raise RuntimeError("LLM Evaluator 실행 전에 .env에 OPENAI_API_KEY를 설정하세요.")
    # Evaluator가 다시 검색하지 않고 Agent가 실제로 받은 검색 결과만 근거로 삼게 합니다.
    search_results = "\n\n".join(
        f"검색어: {detail['arguments'].get('query', '')}\n{detail['output']}"
        for detail in successful_tool_results(run, "wikipedia_search")
        if "output" in detail
    )
    # JSON으로 감싸면 기획서의 줄바꿈과 따옴표가 이스케이프되므로 제목을 붙인 평문으로 전달합니다.
    evaluation_input = (
        f"출처 기준:\n{EVALUATOR_SOURCE_RULES}\n\n"
        f"전시 기획서:\n{read_file('workspace/space_note.md')}\n\n"
        f"실제 위키백과 검색 결과:\n{search_results or '(없음)'}\n\n"
        f"평가할 안내문:\n{run.answer}"
    )
    response = client.responses.create(
        model=MODEL,
        instructions=EVALUATOR_INSTRUCTIONS,
        input=evaluation_input,
        text={"format": EVALUATOR_FORMAT},
    )
    result = json.loads(response.output_text)
    return EvaluatorResult(passed=result["passed"], feedback=result["feedback"].strip())

---
## 2-B.3 Conditional Stop, Feedback, Loop Limit

> **확인 예제:** 실제 루프에서 사용하는 종료·Feedback·반복 한도 함수를 예시 입력으로 먼저 확인합니다.

검증 통과는 전체 Loop를 끝내고, 실패는 부족한 항목을 Feedback으로 만듭니다. 
- `max_iterations`에 도달하면 새 Run을 시작하지 않습니다. 
- 다음 Run은 새 `RunState`를 가지므로 한 Run의 Retry 횟수와 Outer Loop 반복 횟수는 구분됩니다.

**TODO 6~8:** 통과 종료, 누락 항목 Feedback, 반복 한도 판정을 완성하세요.


In [26]:
def loop_limit_reached(iteration: int, max_iterations: int) -> bool:
    """현재 Run 번호가 Outer Loop의 반복 한도에 도달했는지 판단합니다.

    Args:
        iteration: 1부터 시작하는 현재 Run 번호.
        max_iterations: 허용된 최대 Run 횟수.

    Returns:
        한도에 도달했으면 True.
    """
    # TODO 8: 허용된 최대 Run 횟수에 도달했는지 판정하세요.
    return iteration >= max_iterations


def route_loop(verification: Verification, iteration: int, max_iterations: int) -> str:
    """검증 결과와 Run 횟수로 Outer Loop의 다음 행동을 고릅니다.

    Args:
        verification: 이번 Run의 종합 검증 결과.
        iteration: 1부터 시작하는 현재 Run 번호.
        max_iterations: 허용된 최대 Run 횟수.

    Returns:
        "PASSED", "LIMIT_REACHED", "RETRY" 중 하나.
    """
    # TODO 6: 통과 / 한도 도달 / 재실행 중 하나를 고르세요.
    if verification.passed:
        return "PASSED"
    if loop_limit_reached(iteration, max_iterations):
        return "LIMIT_REACHED"
    return "RETRY"


def make_feedback(issues: list[str]) -> str:
    """검증에서 발견한 문제를 다음 Run에 전달할 수정 요청으로 만듭니다.

    Args:
        issues: 검증에서 발견한 문제 목록.

    Returns:
        수정 요청 문장. 문제가 없으면 빈 문자열.
    """
    # TODO 7: issues를 다음 Run이 고칠 수 있는 Feedback 문장으로 만드세요.
    if not issues:
        return ""
    return "다음 문제를 고쳐 안내문을 다시 작성하세요:\n- " + "\n- ".join(issues)


print("통과:", route_loop(Verification(True, []), 1, 3))
print("실패·여유:", route_loop(Verification(False, ["문의처 누락"]), 2, 3))
print("실패·한도:", route_loop(Verification(False, ["문의처 누락"]), 3, 3))
print("Feedback 예시:", make_feedback(["문의처 누락", "위키백과 URL 누락"]))

통과: PASSED
실패·여유: RETRY
실패·한도: LIMIT_REACHED
Feedback 예시: 다음 문제를 고쳐 안내문을 다시 작성하세요:
- 문의처 누락
- 위키백과 URL 누락


---
## 2-B.4 새 Run을 만드는 루프

- `verify_run()`은 Tool Trace·안내문 규칙·LLM 평가를 결합해 한 Run의 완료 조건을 판정합니다. 
- `run_with_feedback()`은 그 결과에 따라 Loop를 끝내거나, 실패 이유와 이전 안내문을 다음 Task에 넣어 **새 Run**을 시작합니다. 
- Run이 오류·Timeout·Max Steps로 끝나면 완성 답변이 없으므로 재검증하지 않고 전체 Loop를 종료합니다.


In [27]:
@dataclass
class LoopResult:
    status: str = "RUNNING"
    runs: list[RunState] = field(default_factory=list)
    verifications: list[Verification] = field(default_factory=list)
    evaluations: list[EvaluatorResult] = field(default_factory=list)
    feedbacks: list[str] = field(default_factory=list)
    inputs: list[str] = field(default_factory=list)


@dataclass
class RunReview:
    trace: Verification
    rules: Verification
    evaluation: EvaluatorResult
    verification: Verification


Evaluator = Callable[[RunState], EvaluatorResult]
AgentRunner = Callable[[str], RunState]


def verify_run(run: RunState, evaluator: Evaluator) -> RunReview:
    """Tool 사용 기록·안내문 규칙·LLM 평가를 종합해 Run을 검증합니다.

    Args:
        run: 검증할 Run 상태.
        evaluator: 안내문 내용을 평가할 함수.

    Returns:
        개별 검사 결과와 종합 검증 결과.
    """
    trace_check = verify_tool_trace(run)
    rule_check = verify_notice(run.answer)
    evaluation = evaluator(run)
    issues = [*trace_check.issues, *rule_check.issues]
    if not evaluation.passed:
        issues.append("LLM 평가: " + (evaluation.feedback or "설명을 다시 검토하세요."))
    return RunReview(trace_check, rule_check, evaluation, Verification(not issues, issues))


def show_review(review: RunReview) -> None:
    """각 검사 결과와 종합 판정을 구분해 출력합니다.

    Args:
        review: 출력할 검증 결과.
    """
    checks = {
        "TOOL TRACE": (review.trace.passed, review.trace.issues),
        "RULES": (review.rules.passed, review.rules.issues),
        "LLM EVALUATOR": (review.evaluation.passed, review.evaluation.feedback),
        "VERIFICATION": (review.verification.passed, review.verification.issues),
    }
    for label, (passed, detail) in checks.items():
        print(f"[{label}]", "PASS" if passed else f"FAIL: {detail}")


def run_with_feedback(
    task: str,
    *,
    agent_runner: AgentRunner = run_agent,
    evaluator: Evaluator = evaluate_notice_with_llm,
    max_iterations: int = 3,
) -> LoopResult:
    """검증에 실패하면 이전 안내문과 Feedback을 담아 새 Run을 만듭니다.

    통과하거나 반복 한도에 도달하면 종료합니다.

    Args:
        task: 원래 사용자 요청.
        agent_runner: 요청 하나로 Run을 실행하는 함수.
        evaluator: 안내문 내용을 평가할 함수.
        max_iterations: 허용된 최대 Run 횟수.

    Returns:
        Run별 입력·결과·검증·Feedback과 최종 상태.
    """
    loop = LoopResult()
    feedback = previous_answer = ""

    for iteration in itertools.count(1):
        current_task = (
            f"{task}\n\n이전 안내문:\n{previous_answer}\n\n검증 피드백:\n{feedback}"
            if feedback else task
        )
        loop.inputs.append(current_task)

        print(f"\n[LOOP {iteration}]")
        run = agent_runner(current_task)
        loop.runs.append(run)
        if run.status != "COMPLETED":
            loop.status = "RUN_FAILED"
            break

        # Evaluator 실패는 안내문 품질 문제가 아니므로 Re-run하지 않고 Loop를 멈춥니다.
        try:
            review = verify_run(run, evaluator)
        except (APIError, TimeoutError, ValueError) as error:
            loop.status = "EVALUATOR_ERROR"
            print("[EVALUATOR ERROR]", type(error).__name__)
            break

        loop.evaluations.append(review.evaluation)
        loop.verifications.append(review.verification)
        show_review(review)

        decision = route_loop(review.verification, iteration, max_iterations)
        if decision != "RETRY":
            loop.status = decision
            print("[LOOP END]" if decision == "PASSED" else "[LOOP STOPPED] MAX_ITERATIONS")
            break

        feedback = make_feedback(review.verification.issues)
        previous_answer = run.answer
        loop.feedbacks.append(feedback)
        print("[FEEDBACK]", feedback)

    return loop

---
## 2-B.5 첫 통과·재실행·반복 한도 관찰

> **모의 시나리오:** 실제 Outer Loop에 미리 정한 안내문·Tool 기록·Evaluator 결과를 전달해 검증·Feedback·재실행·반복 한도를 확인합니다.

모의 Trace에는 성공한 Tool 결과를 넣고, 마지막 사례에서는 검색 기록을 일부러 뺍니다.


In [28]:
def answer_runner(answers: list[str], *, include_wikipedia: bool = True) -> AgentRunner:
    """미리 정한 안내문을 차례로 반환하고 Tool 성공 Trace를 흉내 내는 모의 Agent를 만듭니다.

    Args:
        answers: Run마다 반환할 안내문.
        include_wikipedia: 위키백과 검색 기록을 Trace에 넣을지 여부.

    Returns:
        run_with_feedback의 agent_runner로 쓸 수 있는 함수.
    """
    remaining = iter(answers)
    calls = [("read_file", {"path": path}) for path in REQUIRED_FILES]
    if include_wikipedia:
        calls.append(("wikipedia_search", {"query": "제임스 웹 우주망원경"}))
    tool_trace = [
        {"event": "TOOL_RESULT",
         "detail": {"name": name, "arguments": arguments, "status": "SUCCESS"}}
        for name, arguments in calls
    ]

    def run(task: str) -> RunState:
        print("[RUN INPUT]", task.replace("\n", " ")[:160])
        return RunState(status="COMPLETED", answer=next(remaining), trace=list(tool_trace))

    return run


def scripted_evaluator(results: list[EvaluatorResult]) -> Evaluator:
    """미리 정한 평가 결과를 호출 순서대로 돌려주는 모의 Evaluator를 만듭니다.

    Args:
        results: 차례로 반환할 평가 결과.

    Returns:
        run_with_feedback의 evaluator로 쓸 수 있는 함수.
    """
    remaining = iter(results)
    return lambda _run: next(remaining)


PASS_EVALUATION = EvaluatorResult(True, "")
print("==첫 실행 통과==")
pass_loop = run_with_feedback(
    "전시 안내문 작성",
    agent_runner=answer_runner([SAMPLE_NOTICE]),
    evaluator=scripted_evaluator([PASS_EVALUATION]),
)
print("첫 실행 통과:", pass_loop.status, "| Run", len(pass_loop.runs))

print("\n==규칙 실패 후 재실행==")
improve_loop = run_with_feedback(
    "전시 안내문 작성",
    agent_runner=answer_runner([INCOMPLETE_NOTICE, SAMPLE_NOTICE]),
    evaluator=scripted_evaluator([PASS_EVALUATION, PASS_EVALUATION]),
)
print("실패 뒤 재실행:", improve_loop.status, "| Run", len(improve_loop.runs))
print("두 번째 입력에 전시문의 Feedback 포함:", "전시문의" in improve_loop.inputs[-1])
print("Run 사이에 저장된 Feedback:", f"{len(improve_loop.feedbacks)}개")

print("\n==LLM 평가 실패 후 재실행==")
evaluator_loop = run_with_feedback(
    "전시 안내문 작성",
    agent_runner=answer_runner([SAMPLE_NOTICE, SAMPLE_NOTICE]),
    evaluator=scripted_evaluator([
        EvaluatorResult(
            False, "적외선이 먼지에 가린 천체를 관측하는 데 유용한 이유를 더 쉽게 설명하세요."
        ),
        PASS_EVALUATION,
    ]),
)
print("평가 실패 뒤 재실행:", evaluator_loop.status, "| Run", len(evaluator_loop.runs))
print("두 번째 입력에 평가 Feedback 포함:", "적외선" in evaluator_loop.inputs[-1])
print("Run별 LLM 평가 통과 여부:", [item.passed for item in evaluator_loop.evaluations])

print("\n==반복 한도==")
limit_loop = run_with_feedback(
    "전시 안내문 작성",
    agent_runner=answer_runner([INCOMPLETE_NOTICE] * 3),
    evaluator=scripted_evaluator([PASS_EVALUATION] * 3),
)
print("반복 한도:", limit_loop.status, "| Run", len(limit_loop.runs))

print("\n==필수 Tool 누락==")
missing_tool_loop = run_with_feedback(
    "전시 안내문 작성",
    agent_runner=answer_runner([SAMPLE_NOTICE], include_wikipedia=False),
    evaluator=scripted_evaluator([PASS_EVALUATION]),
    max_iterations=1,
)
print("검색 Tool 누락:", missing_tool_loop.status,
      "| 문제", missing_tool_loop.verifications[0].issues)

==첫 실행 통과==

[LOOP 1]
[RUN INPUT] 전시 안내문 작성
[TOOL TRACE] PASS
[RULES] PASS
[LLM EVALUATOR] PASS
[VERIFICATION] PASS
[LOOP END]
첫 실행 통과: PASSED | Run 1

==규칙 실패 후 재실행==

[LOOP 1]
[RUN INPUT] 전시 안내문 작성
[TOOL TRACE] PASS
[RULES] FAIL: ['필수 문구 또는 관람 정보가 빠졌습니다: - ◯ 전시문의: workshop@example.com', '관람 정보 다섯 항목을 양식 순서대로 쓰세요.', '더 알아보기 줄에 한국어 위키백과 URL을 마크다운 링크 없이 쓰세요. 예: - 🔎 더 알아보기: https://ko.wikipedia.org/wiki/<문서명>']
[LLM EVALUATOR] PASS
[VERIFICATION] FAIL: ['필수 문구 또는 관람 정보가 빠졌습니다: - ◯ 전시문의: workshop@example.com', '관람 정보 다섯 항목을 양식 순서대로 쓰세요.', '더 알아보기 줄에 한국어 위키백과 URL을 마크다운 링크 없이 쓰세요. 예: - 🔎 더 알아보기: https://ko.wikipedia.org/wiki/<문서명>']
[FEEDBACK] 다음 문제를 고쳐 안내문을 다시 작성하세요:
- 필수 문구 또는 관람 정보가 빠졌습니다: - ◯ 전시문의: workshop@example.com
- 관람 정보 다섯 항목을 양식 순서대로 쓰세요.
- 더 알아보기 줄에 한국어 위키백과 URL을 마크다운 링크 없이 쓰세요. 예: - 🔎 더 알아보기: https://ko.wikipedia.org/wiki/<문서명>

[LOOP 2]
[RUN INPUT] 전시 안내문 작성  이전 안내문: # 보이지 않는 빛으로 우주를 보다  눈에 보이지 않는 빛으로 우주를 살펴보세요.  제임스 웹 우주망원경은 2021년에 발사된 적외선 관측 망원경입니다. 적외선은 먼지에 가린 별의 형성 지역과 먼 

> **결과 분석:** 첫 통과 사례는 Run 1개, 규칙 실패와 LLM 평가 실패 사례는 Run 2개에서 `PASSED`로 종료됩니다. 반복 한도 사례는 Run 3개에서, 필수 Tool 누락 사례는 Run 1개에서 `LIMIT_REACHED`로 종료됩니다. `Run 사이에 저장된 Feedback: 1개`는 Run 1의 검증 실패로 만든 Feedback이 `LoopResult`에 남았다는 뜻이고, `Run별 LLM 평가 통과 여부: [False, True]`는 Run마다의 Evaluator 판정이 순서대로 쌓였다는 뜻입니다.


---
## 3. 실제 전시 안내문 Agent 실행

이 단계에서는 실제 OpenAI 모델을 호출합니다. 
- Generator가 기획서·양식·위키백과 정보를 Tool로 모아 안내문을 작성한 뒤, 같은 `MODEL`을 사용하는 LLM Evaluator가 별도의 모델 호출로 안내문을 평가합니다. 
- 검증(규칙·Tool 사용·LLM 평가)에 실패하면 Feedback을 반영하여 새 Run을 시작합니다. 첫 번째 실행 셀에서는 실행 중 로그와 실제 안내문을 확인하고, 다음 셀에서는 저장된 Run별 Trace를 따로 확인합니다. 각 Run의 Trace와 규칙·Tool 사용·LLM Evaluator 결과를 함께 확인하세요. 
- Evaluator 호출로 모델 호출과 비용이 추가됩니다.

> **실행 참고:** 모델의 답변, Tool 호출 순서와 검색 인자, LLM Evaluator의 판정, Feedback 및 Run 횟수는 실행마다 달라질 수 있습니다. 검증 규칙과 조건 분기 구조는 동일합니다.

TODO를 모두 채우고 앞의 재현 결과를 확인한 뒤 `READY_FOR_LIVE_RUN=True`로 바꾸세요.


In [29]:
READY_FOR_LIVE_RUN = True  # TODO를 채운 뒤 True로 변경하세요.

if READY_FOR_LIVE_RUN:
    if client is None:
        raise RuntimeError(".env에 OPENAI_API_KEY를 설정한 뒤 실제 실행을 시작하세요.")
    live_loop = run_with_feedback(TASK, max_iterations=3)
    print("\n[전체 결과]", live_loop.status, f"Run {len(live_loop.runs)}개")
else:
    print("실제 실행을 시작하려면 READY_FOR_LIVE_RUN=True로 설정하세요.")



[LOOP 1]
[RUN START] workspace/space_note.md의 전시 기획서와 templates/exhibition_notice.md의 양식을 바탕으로 제임스 웹 우주망원경 전시 안내문 한 편을 작성해줘. 필요한 사실과 한국어 위키백과 문서 링크를 확인해줘.
[TOOL CALL] read_file({"path":"workspace/space_note.md"})
[TOOL RESULT] read_file: # 제임스 웹 우주망원경 전시 기획서

## 1. 기획 개요
전시 제목은 「보이지 않는 빛으로 우주를 보다」로 정한다. 천문학을 처음 접하는 관람객이 제임스 웹 우주망원경의 이름을 기억하는 데서 그치지 않고, 왜 적외선으로 우주를 관측하는지 이해하도록 돕는 작은 전시다. 주 관람객은 청...
[TOOL CALL] read_file({"path":"templates/exhibition_notice.md"})
[TOOL RESULT] read_file: # 전시 안내문 작성 양식

함께 제공된 전시 기획서는 사실과 운영 정보를 제공합니다. 아래 양식은 안내문의 구성과 출력 순서만 정합니다. 레퍼런스 전시의 이름·기간·장소·연락처는 가져오지 마세요. 기획서의 전시 정보와 이번 단계에서 지정한 외부 참고 문서에서 확인한 내용만 사용하세요....
[TOOL CALL] wikipedia_search({"query":"제임스 웹 우주망원경"})
[TOOL RESULT] wikipedia_search: {"query": "제임스 웹 우주망원경", "results": [{"title": "제임스 웹 우주망원경", "summary": "제임스 웹 우주 망원경(영어: James Webb Space Telescope, JWST)은 적외선 천문 관측을 주목적으로 하는 우주 망원경이다. 현존하는...
[AGENT FINAL ANSWER]
# 보이지 않는 빛으로 우주를 보다

우주에는 우리의 눈에 보이지 않는 수많은 신비가 숨겨져 있습니다.

제임스 웹 우주망원경은 

> **결과 분석:** `[AGENT FINAL ANSWER]`는 각 Run의 안내문이고, `[TOOL TRACE]`·`[RULES]`·`[LLM EVALUATOR]`·`[VERIFICATION]`은 각 검사와 종합 판정입니다. `[전체 결과]`의 `PASSED`는 검증 통과, `LIMIT_REACHED`는 통과하지 못한 상태에서 반복 한도에 도달했음을 뜻합니다.


In [30]:
if "live_loop" in globals():
    for index, run in enumerate(live_loop.runs, 1):
        print(f"\n[RUN {index} TRACE]")
        show_trace(run)
else:
    print("위 셀에서 실제 실행을 먼저 완료하세요.")


[RUN 1 TRACE]
status=COMPLETED, steps=2, tool_calls=3, retries=0
01. RUN_START: 
02. MODEL_CALL: step=1
03. TOOL_CALL: read_file
04. TOOL_RESULT: {'name': 'read_file', 'arguments': {'path': 'workspace/space_note.md'}, 'output': '# 제임스 웹 우주망원경 전시 기획서\n\n## 1. 기획 개요\n전시 제목은 「보이지 않는 빛으로 우주를 보다」로 정한다. 천문학을 처음 접하는 관람객이 제임스 웹 우주망원경의 이름을 기억하는 데서 그치지 않고, 왜 적...', 'status': 'SUCCESS'}
05. TOOL_CALL: read_file
06. TOOL_RESULT: {'name': 'read_file', 'arguments': {'path': 'templates/exhibition_notice.md'}, 'output': '# 전시 안내문 작성 양식\n\n함께 제공된 전시 기획서는 사실과 운영 정보를 제공합니다. 아래 양식은 안내문의 구성과 출력 순서만 정합니다. 레퍼런스 전시의 이름·기간·장소·연락처는 가져오지 마세요. 기획서의 전시 정...', 'status': 'SUCCESS'}
07. TOOL_CALL: wikipedia_search
08. TOOL_RESULT: {'name': 'wikipedia_search', 'arguments': {'query': '제임스 웹 우주망원경'}, 'output': '{"query": "제임스 웹 우주망원경", "results": [{"title": "제임스 웹 우주망원경", "summary": "제임스 웹 우주 망원경(영어: James Webb Space Telescope, J...', 'status': 'SUCCESS'}
09. MODEL_CALL: step=2
10. FINAL_ANSWER_RECORDED: characters=511

> **결과 분석:** `[RUN n TRACE]`는 각 Run에 저장된 이벤트입니다. `TOOL_RESULT`의 `output`은 출력할 때만 앞 120자로 줄이며, Trace에는 전문이 저장되어 LLM Evaluator가 사용합니다. `MODEL_CALL: step=n`은 해당 Run 안의 모델 호출 번호이며, `FINAL_ANSWER_RECORDED: characters=n`은 저장한 답변의 글자 수입니다. `RUN_END`는 그 Run의 정상 종료를 뜻하며 전체 검증 통과를 뜻하지는 않습니다.


> **고찰:** 각 Run의 정상 종료와 전체 결과의 검증 통과는 서로 다른 판단입니다. 새로운 Run은 이전 안내문과 Feedback을 입력으로 받지만 실행 상태와 Trace는 새로 생성합니다.


---
## 전체 정리

Agent Loop는 모델이 다음 행동을 고릅니다. Harness는 한 Run의 횟수·오류·Timeout과 Trace를 관리합니다. Outer Loop는 필수 Tool 사용 Trace, 규칙, LLM Evaluator를 확인하고 Feedback을 반영해 새 Run을 시작합니다. 실습 3에서는 같은 개념을 LangChain과 LangGraph의 구조로 비교합니다.

참고: [OpenAI Function Calling](https://developers.openai.com/api/docs/guides/function-calling), [OpenAI Python SDK Retry·Timeout](https://github.com/openai/openai-python#retries), [NASA Webb Fact Sheet](https://science.nasa.gov/mission/webb/fact-sheet/)
